# OMNIS-COURT LLM Server v7.12 (128K Context)
## Qwen3-8B + 128K Context (via YaRN RoPE Scaling)

### Instructions:
1. Runtime -> Factory reset runtime
2. Runtime -> Change runtime type -> T4 GPU
3. Run All (Ctrl+F9)
4. Wait 5-8 minutes for model loading
5. Copy LLM + Jina URLs from Cell 5
6. Paste into config/platforms.json
7. Close tab (anti-idle is active)

### Architecture:
- Model: Qwen3-8B (4-bit quantization, ~6GB)
- Context: 128K tokens (via YaRN RoPE scaling 3.2x)
- Server: FastAPI + OpenAI-compatible
- Total VRAM: ~7-8GB / 15GB T4

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Core packages (no vLLM - stable)
!pip install -q bitsandbytes accelerate trafilatura fastapi uvicorn nest-asyncio requests

# Install cloudflared binary
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify installations
import subprocess
cf = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'\ncloudflared: {cf.stdout.strip()}')

all_ok = True
for pkg in ['torch', 'transformers', 'bitsandbytes', 'accelerate', 'trafilatura', 'fastapi', 'uvicorn']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'OK {pkg}')
    except Exception as e:
        print(f'FAIL {pkg}: {e}')
        all_ok = False

if all_ok:
    print('\nALL dependencies ready!')
else:
    print('\nSOME packages failed. STOP here and send error.')

In [ ]:
# ============================================================
# CELL 2: ANTI-IDLE
# ============================================================
from IPython.display import display, Javascript

display(Javascript('''
    setInterval(function(){
        var btn = document.querySelector('colab-run-button');
        if(btn) btn.click();
    }, 300000);
'''))
print('Anti-idle active! Safe to close tab after all cells run.')

In [ ]:
# ============================================================
# CELL 3: LOAD QWEN3-8B + 128K CONTEXT (YaRN RoPE Scaling)
# ============================================================
import torch
import time
import threading
import requests
import json
import hashlib
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig
from fastapi import FastAPI
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio
nest_asyncio.apply()

print('Loading Qwen3-8B with 4-bit quantization + 128K context...')
print('   - Model: Qwen3-8B (4-bit)')
print('   - Context: 128K tokens (via YaRN RoPE scaling 3.2x)')
print('   - Expected VRAM: ~7-8 GB / 15 GB')
print('   - Expected load time: 5-8 minutes')
print()

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

MODEL_ID = 'Qwen/Qwen3-8B'
TARGET_CONTEXT = 131072  # 128K tokens

# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True
)
print('Tokenizer loaded')

# Load config and modify for 128K
print('Loading config with 128K context scaling...')
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)

# Apply YaRN RoPE scaling for 128K (40K * 3.2 = 128K)
# Qwen3 uses YaRN (Yet another RoPE extensioN) for context extension
config.rope_scaling = {
    "rope_type": "yarn",
    "factor": 3.2,
    "original_max_position_embeddings": 40960
}
config.max_position_embeddings = TARGET_CONTEXT

print(f'Config modified: {config.max_position_embeddings} tokens')

# Load model with modified config
print('Loading model (this takes 5-8 min)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
print('Model loaded successfully with 128K context!')

# Check memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1024**3
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {mem_used:.1f}GB / {mem_total:.1f}GB ({100*mem_used/mem_total:.1f}%)')

MAX_CONTEXT_LEN = TARGET_CONTEXT
print(f'Max context: {MAX_CONTEXT_LEN} tokens')

# Store globals for server
GLOBAL_MODEL = model
GLOBAL_TOKENIZER = tokenizer
GLOBAL_MAX_CONTEXT = MAX_CONTEXT_LEN

# ==========================================
# OpenAI-compatible FastAPI server
# ==========================================
app = FastAPI(title='OMNIS Qwen3-8B Server (128K)')

@app.get('/health')
async def health():
    return {'status': 'ok', 'model': 'qwen3-8b-128k'}

@app.get('/v1/models')
async def list_models():
    return {
        'data': [{
            'id': 'qwen3',
            'object': 'model',
            'created': int(time.time()),
            'owned_by': 'local'
        }]
    }

@app.post('/v1/chat/completions')
async def chat_completions(request: dict):
    try:
        messages = request.get('messages', [])
        max_tokens = request.get('max_tokens', 8192)
        temperature = request.get('temperature', 0.7)
        top_p = request.get('top_p', 0.9)
        
        # Handle Qwen3 thinking mode (disable for speed)
        if messages and messages[0].get('role') == 'user':
            content = messages[0].get('content', '')
            if not content.startswith('/no_think'):
                messages[0]['content'] = '/no_think\n' + content
        
        text = GLOBAL_TOKENIZER.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = GLOBAL_TOKENIZER(
            text, 
            return_tensors='pt',
            truncation=True,
            max_length=GLOBAL_MAX_CONTEXT
        ).to(GLOBAL_MODEL.device)
        
        input_len = inputs['input_ids'].shape[1]
        
        with torch.no_grad():
            outputs = GLOBAL_MODEL.generate(
                **inputs,
                max_new_tokens=min(max_tokens, GLOBAL_MAX_CONTEXT - input_len - 100),
                temperature=temperature,
                top_p=top_p,
                do_sample=True if temperature > 0 else False,
                pad_token_id=GLOBAL_TOKENIZER.eos_token_id
            )
        
        new_tokens = outputs[0][input_len:]
        response_text = GLOBAL_TOKENIZER.decode(new_tokens, skip_special_tokens=True)
        
        # Strip thinking tags if present
        if '' in response_text:
            import re
            response_text = re.sub(
                r'',
                '',
                response_text,
                flags=re.DOTALL
            ).strip()
        
        return {
            'id': f'chatcmpl-{int(time.time())}',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': 'qwen3',
            'choices': [{
                'index': 0,
                'message': {
                    'role': 'assistant',
                    'content': response_text
                },
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': input_len,
                'completion_tokens': int(len(new_tokens)),
                'total_tokens': input_len + len(new_tokens)
            }
        }
    except Exception as e:
        print(f'Server error: {e}')
        import traceback
        traceback.print_exc()
        return JSONResponse(status_code=500, content={'error': str(e)})

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

for i in range(30):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'\nLLM Server READY on port 8000 ({(i+1)}s)')
            print('Configuration:')
            print(f'   - Model: Qwen3-8B (4-bit)')
            print(f'   - Context: {MAX_CONTEXT_LEN} tokens (128K!)')
            break
    except:
        pass
    time.sleep(1)
else:
    print('LLM Server failed to start')

In [ ]:
# ============================================================
# CELL 4: START JINA READER SERVER
# ============================================================
import threading, time, requests as req
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn

jina_app = FastAPI(title='OMNIS Jina Reader')

@jina_app.get('/health')
async def jina_health():
    return {'status':'ok'}

@jina_app.get('/extract')
async def extract(url: str = Query(...)):
    try:
        dl = trafilatura.fetch_url(url)
        if not dl:
            return JSONResponse(400, content={'error':'fetch failed','url':url})
        txt = trafilatura.extract(dl, include_comments=False, include_tables=True, no_fallback=False)
        if not txt or len(txt.strip()) < 50:
            return JSONResponse(400, content={'error':'content too short','url':url})
        return {'url':url,'content':txt,'word_count':len(txt.split()),'status':'success'}
    except Exception as e:
        return JSONResponse(500, content={'error':str(e),'url':url})

def run_jina():
    uvicorn.run(jina_app, host='0.0.0.0', port=8001, log_level='warning')

jina_thread = threading.Thread(target=run_jina, daemon=True)
jina_thread.start()
time.sleep(3)

try:
    r = req.get('http://localhost:8001/health', timeout=5)
    print('Jina Reader READY on port 8001' if r.status_code==200 else 'Jina error')
except Exception as e:
    print(f'Jina failed: {e}')

In [ ]:
# ============================================================
# CELL 5: CLOUDFLARE TUNNELS
# ============================================================
import subprocess, re

def tunnel(port):
    p = subprocess.Popen(
        ['cloudflared','tunnel','--url',f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return p, m.group(0)
    return p, None

print('Tunnel LLM (8000)...')
p1, u1 = tunnel(8000)
print('Tunnel Jina (8001)...')
p2, u2 = tunnel(8001)

if u1 and u2:
    print('\n' + '='*60)
    print('OMNIS-COURT COLAB READY! (v7.12 - 128K Context)')
    print('='*60)
    print(f'LLM:  {u1}')
    print(f'JINA: {u2}')
    print('='*60)
    print('COPY BOTH URLs -> config/platforms.json')
    print('Anti-idle ON -> safe to close tab')
    print('Test URLs from YOUR browser (not from Colab)')
    print('='*60)
else:
    print(f'Tunnel failed: LLM={u1}, Jina={u2}')

In [ ]:
# ============================================================
# CELL 6: LOCALHOST TESTS
# ============================================================
import requests

print('Testing LLM on localhost:8000...')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={'model':'qwen3','messages':[{'role':'user','content':'Say OK if you are Qwen3-8B with 128K context'}],'max_tokens':50,'temperature':0.7},
        timeout=120
    )
    if r.status_code == 200:
        resp = r.json()['choices'][0]['message']['content']
        print(f'LLM localhost OK: {resp[:100]}')
    else:
        print(f'LLM localhost: {r.status_code} - {r.text[:200]}')
except Exception as e:
    print(f'LLM localhost: {e}')

print('\nTesting Jina on localhost:8001...')
try:
    r = requests.get(
        'http://localhost:8001/extract',
        params={'url':'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        print(f"Jina localhost OK: {r.json()['word_count']} words")
    else:
        print(f'Jina localhost: {r.status_code}')
except Exception as e:
    print(f'Jina localhost: {e}')

print('\n' + '='*60)
print('NOW TEST TUNNEL URLs FROM YOUR BROWSER:')
print(f'   {u1}/v1/models')
print(f'   {u2}/health')
print('='*60)